<a href="https://colab.research.google.com/github/PolPetr/python-course/blob/main/ML/%D0%9A%D0%BE%D0%BF%D0%B8%D1%8F_%D0%B1%D0%BB%D0%BE%D0%BA%D0%BD%D0%BE%D1%82%D0%B0_%22log_reg_hw_ipynb%22_%D0%9F%D0%B5%D1%82%D1%80%D0%BE%D0%B2%D0%B0.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Домашнее задание: линейные модели для NLP**

## **Что будем делать?**
Мы будем предсказывать, к какой категории относится новость: про хоккей или про космос.

## **Часть 1. Практическая работа (8 баллов)**

### **1. Загрузка данных**

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import fetch_20newsgroups

# Загружаем данные о новостях
categories = ['rec.sport.hockey', 'sci.space']
newsgroups = fetch_20newsgroups(subset='all',
                               categories=categories,
                               shuffle=True,
                               random_state=42)

# Создаём таблицу с данными
data = pd.DataFrame({
    'text': newsgroups.data,
    'category': newsgroups.target  # 0 = хоккей, 1 = космос
})

print("Размер данных:", data.shape)
print("\nПервые 3 текста:")
for i in range(3):
    print(f"Текст {i+1}: {data['text'][i][:100]}...")

Размер данных: (1986, 2)

Первые 3 текста:
Текст 1: From: mccall@mksol.dseg.ti.com (fred j mccall 575-3539)
Subject: Re: Vandalizing the sky.
Article-I....
Текст 2: From: epritcha@s.psych.uiuc.edu ( Evan Pritchard)
Subject: Re: div. and conf. names
Distribution: na...
Текст 3: From: baalke@kelvin.jpl.nasa.gov (Ron Baalke)
Subject: Galileo Update - 04/29/93
Keywords: Galileo, ...


**Задание 1:**
- Сколько всего текстов в датасете?
- Выведите количество текстов в каждой категории

In [2]:
print("Текстов в датасете: ", data.shape[0])
print("Количество категорий: ", data.shape[1])
print("\nКоличество текстов в каждой категории: ", data['category'].value_counts())

Текстов в датасете:  1986
Количество категорий:  2

Количество текстов в каждой категории:  category
0    999
1    987
Name: count, dtype: int64


### **2. Анализ данных**

In [5]:
counts = data['category'].value_counts()
if counts[0] > counts[1]:
  print("Категория 'хоккей' больше")
elif counts[1] > counts[0]:
  print("Категория 'космос' больше")
else:
  print("Категории равны")

Категория 'хоккей' больше


**Вопрос:** Какая категория больше: хоккей или космос?

### **3. Очистка текста**

In [10]:
import re

def clean_text(text):
    text = text.lower()
    text = re.sub(r'[^a-zA-Z\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text)
    # 1. Привести текст к нижнему регистру (.lower())
    # 2. Удалить всё, кроме букв и пробелов (re.sub)
    # 3. Убрать лишние пробелы
    return text

# Применяем функцию
data['clean_text'] = data['text'].apply(clean_text)

# Проверяем
print("До очистки:", data['text'][0][:100])
print("После очистки:", data['clean_text'][0][:100])

До очистки: From: mccall@mksol.dseg.ti.com (fred j mccall 575-3539)
Subject: Re: Vandalizing the sky.
Article-I.
После очистки: from mccall mksol dseg ti com fred j mccall subject re vandalizing the sky article i d mksol apr org


### **4. Создание признаков (векторизация)**

In [11]:
from sklearn.feature_extraction.text import CountVectorizer

# Ваш код здесь
# 1. Создайте CountVectorizer (остановитесь на английском)
# 2. Преобразуйте тексты в числа (fit_transform)
# 3. Выведите сколько слов получилось
vectorizer = CountVectorizer(stop_words='english')
X = vectorizer.fit_transform(data['clean_text'])
print("Количество полученных слов", X.shape)

Количество полученных слов (1986, 24480)


**Вопрос:** Сколько уникальных слов нашёл CountVectorizer?
**Ответ**: 24 480 слов

### **5. Разделение данных**

In [12]:
from sklearn.model_selection import train_test_split

# Цель: предсказать категорию (0 или 1)
X = X
y = data['category']

# Ваш код здесь
# Разделите данные на 80% обучение, 20% тест
# Используйте random_state=42
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

### **6. Обучение модели**

In [15]:
from sklearn.linear_model import LogisticRegression

# Ваш код здесь
# 1. Создайте модель логистической регрессии
# 2. Обучите её на обучающих данных
# 3. Сделайте предсказания для тестовых данных
model = LogisticRegression()
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

### **7. Оценка модели**

In [16]:
from sklearn.metrics import accuracy_score

# Ваш код здесь
# 1. Вычислите accuracy (точность) модели
# 2. Выведите результат в процентах

# Оценка модели
accuracy = accuracy_score(y_test, y_pred)
print(f"Точность модели составляет {accuracy: .2%}")

Точность модели составляет  99.25%


**Вопрос:** Какая точность у вашей модели? Что это значит?
**Ответ**: Точность высокая (99, 25%, то есть почти 100%). Это значит, что модель практически безошибочно присваивает категорию (хоккей/космос)

### **8. Анализ ошибок**

In [20]:
from sklearn.metrics import confusion_matrix

# Ваш код здесь
cm = confusion_matrix(y_test, y_pred)
print("Матрица ошибок: \n", cm)

# БОНУС: визуализируйте матрицу ошибок
plt.figure(figsize=(6, 4))
# Ваш код для визуализации здесь
plt.show()

Матрица ошибок: 
 [[201   1]
 [  2 194]]


<Figure size 600x400 with 0 Axes>

**Вопрос:** Сколько текстов про космос модель приняла за хоккей?
**Ответ**: По матрице ошибок получается, что 2 текста про космос были ошибочно приняты за хоккей.